In [1]:
import os
os.chdir("/Users/samswitz/GitHub/mech-int")

In [8]:
# os.listdir()

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import transformer_lens
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import tqdm
from functools import partial

/Users/samswitz/miniforge3/envs/sae/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from mechint.sae import SparseAutoEncoder

In [4]:
model = transformer_lens.HookedTransformer.from_pretrained("pythia-70m")

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model pythia-70m into HookedTransformer


In [5]:
EXPANSION = 4
MODEL_PATH = f"saved_models/sae_model_{EXPANSION}x.pt"

# Load saved SAE
d = 512
m = d * EXPANSION

SAE = SparseAutoEncoder(d=d, m=m)
SAE.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
SAE.eval()

device = model.cfg.device
SAE.to(device)

SparseAutoEncoder(
  (act): ReLU()
)

In [6]:
def ablation_hook(resid, hook, feature_idx):
    f = SAE.encoder(resid)
    contr = f[..., feature_idx].unsqueeze(-1) * SAE.W_dec.data[feature_idx]
    return resid - contr

In [55]:
# text = "The governor of New York announced legislation to improve public transportation in the state."

# text = "The grass in the forest"
# text = "We had fun."
text = "I went to the store, and then I went home."

# text = "The grass in the forest"
# text = "The land"

In [31]:
IDX = 1273

ablated_logits = model.run_with_hooks(
    text,
    fwd_hooks=[("blocks.3.hook_resid_post", partial(ablation_hook, feature_idx=IDX))]
)

In [32]:
clean_logits = model(text)

In [33]:
clean_logprobs = F.log_softmax(clean_logits.float(), dim=-1)
ablated_logprobs = F.log_softmax(ablated_logits.float(), dim=-1)
clean_probs = clean_logprobs.exp()

kl_div = (clean_probs * (clean_logprobs - ablated_logprobs)).sum(dim=-1)
print(f"KL divergence per position: {kl_div.squeeze()}")
print(f"Mean KL divergence: {kl_div.mean().item():.6f}")

clean_top = clean_logits.argmax(dim=-1).squeeze()
ablated_top = ablated_logits.argmax(dim=-1).squeeze()

tokenizer = model.tokenizer
for pos in range(len(clean_top)):
    if clean_top[pos] != ablated_top[pos]:
        print(f"Position {pos}: {tokenizer.decode(clean_top[pos])} → {tokenizer.decode(ablated_top[pos])}")

KL divergence per position: tensor([5.9609e+00, 1.1327e-01, 7.4234e-02, 7.1661e-02, 8.4168e-02, 2.3695e-02,
        2.8071e-02, 1.8183e-02, 2.2458e-02, 4.3003e-02, 1.5261e-02, 3.0031e-03,
        4.2668e+00], device='mps:0', grad_fn=<SqueezeBackward0>)
Mean KL divergence: 0.824973
Position 0: Q → .
Position 1: 'm →  have
Position 9:  got →  went
Position 12:  I →  again


In [34]:
tokens = model.to_str_tokens(text)

# KL per token position
print("KL divergence per token:\n")
for pos, (tok, kl) in enumerate(zip(tokens, kl_div.squeeze())):
    print(f"{pos:2d} {tok:>15s}  KL={kl:.6f}")

# Top-5 predictions comparison
k = 5
print("\nTop-5 predictions (clean vs ablated):\n")
for pos in range(len(tokens)):
    clean_topk = clean_logits[0, pos].topk(k)
    ablated_topk = ablated_logits[0, pos].topk(k)
    print(f"Position {pos}: '{tokens[pos]}'")
    print(f"  Clean:   {[tokenizer.decode(t) for t in clean_topk.indices]}")
    print(f"  Ablated: {[tokenizer.decode(t) for t in ablated_topk.indices]}")
    print()

KL divergence per token:

 0   <|endoftext|>  KL=5.960871
 1               I  KL=0.113269
 2            went  KL=0.074234
 3              to  KL=0.071661
 4             the  KL=0.084168
 5           store  KL=0.023695
 6               ,  KL=0.028071
 7             and  KL=0.018183
 8            then  KL=0.022458
 9               I  KL=0.043003
10            went  KL=0.015261
11            home  KL=0.003003
12               .  KL=4.266776

Top-5 predictions (clean vs ablated):

Position 0: '<|endoftext|>'
  Clean:   ['Q', 'The', '\n', 'A', '1']
  Ablated: ['.', ',', ' in', ' the', ' a']

Position 1: 'I'
  Clean:   ["'m", ' have', '’', ' am', ' don']
  Ablated: [' have', "'m", '’', "'ve", ' was']

Position 2: ' went'
  Clean:   [' to', ' through', ' for', ' into', ' on']
  Ablated: [' to', ' into', ' on', ' through', ' out']

Position 3: ' to'
  Clean:   [' the', ' a', ' work', ' school', ' my']
  Ablated: [' the', ' work', ' a', ' my', ' see']

Position 4: ' the'
  Clean:   [' office', 

In [35]:
# Greedy generation: clean vs ablated
def generate_with_ablation(text, feature_idx, max_new_tokens=5):
    clean_tokens = [text]
    ablated_tokens = [text]

    clean_input = text
    ablated_input = text

    for _ in range(max_new_tokens):
        clean_logits = model(clean_input)
        next_clean = tokenizer.decode(clean_logits[0, -1].argmax())
        clean_input += next_clean
        clean_tokens.append(next_clean)

        ablated_logits = model.run_with_hooks(
            ablated_input,
            fwd_hooks=[("blocks.3.hook_resid_post", partial(ablation_hook, feature_idx=feature_idx))]
        )
        next_ablated = tokenizer.decode(ablated_logits[0, -1].argmax())
        ablated_input += next_ablated
        ablated_tokens.append(next_ablated)

    # print(f"Clean:   {clean_input}")
    # print(f"Ablated: {ablated_input}")
    return {"clean": clean_input, "ablated": ablated_input}

# generate_with_ablation(text, IDX)

### 1273 vs 1315 vs 476
- three token recongizers ('.')
- all have causual impact, although 1273 seems to impact the distribution of generated tokens more
- interstingly, cosine similarity is only 0.09 for 1273 and 1315, which is slightly more than 1 SD from mean of ~0
- however, 476 and 1315 have the highest cosine similarity in the entire set of features!
    - makes sense given that the ablated generation for each feature is the same
- 476 and 1273 are marginally more similar at ~0.12 cosine similarity

$\implies$ so the two somewhat causal features (1315, 476) have high similarity with each other, and both have a similar similarity score to the most causal feature (1273) about 1 SD above the mean

In [106]:
# text = "We went to the store. Then we"
# text = "That was incredible!"
text = "Dr. Smith arrived late."

generate_with_ablation(text, 1273, max_new_tokens=10)

{'clean': 'Dr. Smith arrived late. He was a great friend of mine. He was',
 'ablated': 'Dr. Smith arrived late.\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n'}

In [107]:
generate_with_ablation(text, 1315, max_new_tokens=10)

{'clean': 'Dr. Smith arrived late. He was a great friend of mine. He was',
 'ablated': 'Dr. Smith arrived late. He was a great friend of mine. He was'}

In [108]:
generate_with_ablation(text, 476, max_new_tokens=10)

{'clean': 'Dr. Smith arrived late. He was a great friend of mine. He was',
 'ablated': 'Dr. Smith arrived late. He was a great friend of mine. He was'}

### Multi-Ablation

In [109]:
def ablation_hook_multi(resid, hook, feature_idxs):
    f = SAE.encoder(resid)
    idxs = torch.tensor(feature_idxs, device=resid.device)

    # Sum f_i * W_dec[i] across selected features
    contributions = f[..., idxs] @ SAE.W_dec[idxs]

    return resid - contributions

In [110]:
FEATURES = [1273, 1315, 476]

ablated_logits = model.run_with_hooks(
    text,
    fwd_hooks=[
        ("blocks.3.hook_resid_post", partial(ablation_hook_multi, feature_idxs=FEATURES))
    ],
)

In [111]:
def generate_with_multi_ablation(text, feature_idxs, max_new_tokens=5):
    clean_input = text
    ablated_input = text

    for _ in range(max_new_tokens):
        clean_logits = model(clean_input)
        next_clean = tokenizer.decode(clean_logits[0, -1].argmax())
        clean_input += next_clean

        ablated_logits = model.run_with_hooks(
            ablated_input,
            fwd_hooks=[
                ("blocks.3.hook_resid_post", partial(ablation_hook_multi, feature_idxs=feature_idxs))
            ],
        )
        next_ablated = tokenizer.decode(ablated_logits[0, -1].argmax())
        ablated_input += next_ablated

    return {"clean": clean_input, "ablated": ablated_input}

In [112]:
generate_with_multi_ablation(
    text,
    [1273, 1315, 476],
    # [1273],
    max_new_tokens=10,
)

{'clean': 'Dr. Smith arrived late. He was a great friend of mine. He was',
 'ablated': 'Dr. Smith arrived late.\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n'}

In [113]:
def compare_multi_ablation_kl(text, feature_idxs, k=5):
    clean_logits = model(text)
    ablated_logits = model.run_with_hooks(
        text,
        fwd_hooks=[
            ("blocks.3.hook_resid_post", partial(ablation_hook_multi, feature_idxs=feature_idxs))
        ],
    )

    clean_logprobs = F.log_softmax(clean_logits.float(), dim=-1)
    ablated_logprobs = F.log_softmax(ablated_logits.float(), dim=-1)
    clean_probs = clean_logprobs.exp()
    kl_div = (clean_probs * (clean_logprobs - ablated_logprobs)).sum(dim=-1)

    tokens = model.to_str_tokens(text)
    print(f"Features ablated: {feature_idxs}")
    print(f"Mean KL divergence: {kl_div.mean().item():.6f}")
    print("KL divergence per token:\n")
    for pos, (tok, kl) in enumerate(zip(tokens, kl_div.squeeze())):
        print(f"{pos:2d} {tok:>15s}  KL={kl:.6f}")

    print("\nTop-5 predictions (clean vs ablated):\n")
    for pos in range(len(tokens)):
        clean_topk = clean_logits[0, pos].topk(k)
        ablated_topk = ablated_logits[0, pos].topk(k)
        print(f"Position {pos}: '{tokens[pos]}'")
        print(f"  Clean:   {[tokenizer.decode(t) for t in clean_topk.indices]}")
        print(f"  Ablated: {[tokenizer.decode(t) for t in ablated_topk.indices]}")
        print()

    return {
        "clean_logits": clean_logits,
        "ablated_logits": ablated_logits,
        "kl_div": kl_div,
        "mean_kl": kl_div.mean().item(),
    }


_ = compare_multi_ablation_kl(
    text,
    [1273, 1315, 476],
)

Features ablated: [1273, 1315, 476]
Mean KL divergence: 1.126281
KL divergence per token:

 0   <|endoftext|>  KL=5.960866
 1              Dr  KL=0.169961
 2               .  KL=1.497618
 3           Smith  KL=0.109328
 4         arrived  KL=0.028833
 5            late  KL=0.040564
 6               .  KL=0.076796

Top-5 predictions (clean vs ablated):

Position 0: '<|endoftext|>'
  Clean:   ['Q', 'The', '\n', 'A', '1']
  Ablated: ['.', ',', ' in', ' the', ' a']

Position 1: 'Dr'
  Clean:   ['.', 'inking', 'one', 'ama', ' John']
  Ablated: ['.', 'inking', 'one', ' Dr', 'ama']

Position 2: '.'
  Clean:   [' Dr', '\n', ' I', ' K', ' David']
  Ablated: [' and', ' Dr', ' is', ' at', ' the']

Position 3: ' Smith'
  Clean:   ['\n', ',', "'s", ' (', ' is']
  Ablated: [',', '\n', ' (', ' is', '’']

Position 4: ' arrived'
  Clean:   [' at', ' in', ' on', ' to', ' from']
  Ablated: [' at', ' in', ' on', ' to', ' from']

Position 5: ' late'
  Clean:   [' last', ' on', ' this', ' in', ',']
  Ablate

In [114]:
_ = compare_multi_ablation_kl(
    text,
    [1315, 476],
)

Features ablated: [1315, 476]
Mean KL divergence: 0.306682
KL divergence per token:

 0   <|endoftext|>  KL=0.000000
 1              Dr  KL=0.000000
 2               .  KL=2.143950
 3           Smith  KL=0.001687
 4         arrived  KL=0.000024
 5            late  KL=0.000078
 6               .  KL=0.001037

Top-5 predictions (clean vs ablated):

Position 0: '<|endoftext|>'
  Clean:   ['Q', 'The', '\n', 'A', '1']
  Ablated: ['Q', 'The', '\n', 'A', '1']

Position 1: 'Dr'
  Clean:   ['.', 'inking', 'one', 'ama', ' John']
  Ablated: ['.', 'inking', 'one', 'ama', ' John']

Position 2: '.'
  Clean:   [' Dr', '\n', ' I', ' K', ' David']
  Ablated: ['\n', 'Q', 'D', 'A', 'M']

Position 3: ' Smith'
  Clean:   ['\n', ',', "'s", ' (', ' is']
  Ablated: ['\n', ',', "'s", ' (', ' is']

Position 4: ' arrived'
  Clean:   [' at', ' in', ' on', ' to', ' from']
  Ablated: [' at', ' in', ' on', ' to', ' from']

Position 5: ' late'
  Clean:   [' last', ' on', ' this', ' in', ',']
  Ablated: [' last', ' on